# Optogenetics — Visualization

This notebook loads the previously built combined DataFrame (see `optogenetics_build_dataframe.ipynb`) and runs the visualization / exploration workflows.

In [ ]:
# =============================================================================
# 1. ENVIRONMENT SETUP & MODULE IMPORTS
# =============================================================================
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

print(f"✅ Analysis modules loaded from: {MODULE_PATH}")

In [ ]:
from nwb_utils import NWBUtils
from optogenetics_utils import (
    create_opto_data_frame,
    find_unique_combinations,
    find_unique_stimulation,
    load_opto_data_frame,
    find_unique_values_by_conditions,
    normalize_string_columns,
)
from optogenetics_visualization import (
    plot_stay_switch_over_window,
    plot_rates_vs_latent,
)

## Load the pre-built combined DataFrame

In [ ]:
combined_dataframe = load_opto_data_frame(
    csv_path='/root/capsule/scratch/combined_opto_data_frame.csv'
)
combined_dataframe = normalize_string_columns(combined_dataframe)
combined_dataframe.shape

## Inspect unique stimulation conditions

In [ ]:
find_unique_stimulation(
    combined_dataframe,
    columns=[
        'laser_on_trial', 'laser_wavelength', 'laser_location',
        'laser_duration', 'laser_start', 'laser_start_offset',
        'laser_end', 'laser_end_offset', 'laser_protocol',
        'laser_frequency', 'laser_pulse_duration',
        'session_wide_control', 'fraction_of_session', 'session_start_with',
        'session_alternation',
        'laser_1_target_areas', 'laser_2_target_areas', 'laser_rampingdown',
    ],
    include_id_lists=False,
    include={"laser_1_target_areas":["left VP GABAergic neuron inactivation"],"laser_2_target_areas":["right VP GABAergic neuron inactivation"]}
    
)

In [ ]:
conds = {"laser_start": "Go cue", "laser_end": "Trial start"}
unique_vals = find_unique_values_by_conditions(combined_dataframe, conds, output_column="session")
print(unique_vals)

## Build a `criteria` dict from one stimulation row

In [ ]:
import pandas as pd

conditions = find_unique_stimulation(
    combined_dataframe,
    columns=[
        'laser_on_trial', 'laser_wavelength', 'laser_location',
        'laser_duration', 'laser_start', 'laser_start_offset',
        'laser_end', 'laser_end_offset', 'laser_protocol',
        'laser_frequency', 'laser_pulse_duration',
        'session_wide_control', 'fraction_of_session', 'session_start_with',
        'session_alternation',
        'laser_1_target_areas', 'laser_2_target_areas', 'laser_rampingdown',
    ],
    include_id_lists=False,
    include={
        "laser_1_target_areas": ["left VP GABAergic neuron inactivation"],
        "laser_2_target_areas": ["right VP GABAergic neuron inactivation"],
    },
)

conditions = conditions.sort_values(
    by=["laser_1_target_areas", "laser_2_target_areas", "session_wide_control", "n_trials", "laser_start"],
    ascending=[True, False, False, False, False],
)

row = conditions.loc[5]
criteria = (
    row.drop(labels=['n_trials', 'n_session', 'n_mice'], errors='ignore')
       .where(pd.notna(row), None)
       .to_dict()
)
criteria

In [ ]:
conditions

## Plot rates vs. a latent variable (ITI delay sum)

In [ ]:
figs = plot_rates_vs_latent(
    combined_dataframe,
    latent_col="ITI_delay_sum",
    line_by="all",
    bins=9,
    binning="uniform",
    criteria=criteria,
    latent_range=(0, 15),
    window=[-1, 2],
)

## Stay/switch over window + rates vs. deltaQ

In [ ]:
plot_stay_switch_over_window(
    combined_dataframe,
    criteria=criteria,
    window=[-2, 2],
    line_by='subject',
)

figs = plot_rates_vs_latent(
    combined_dataframe,
    latent_col="QLearning_L1F1_CK1_softmax-deltaQ-1",
    line_by="all",
    bins=8,
    binning="uniform",
    criteria=criteria,
    window=[-1, 2],
)